# EPCLM-002-A0 — Persistent State Recall

One-click A0-v0.2 screening run. Enable Internet, select a GPU accelerator, and add the existing Kaggle Secret `GITHUB_TOKEN`, then use **Run All** once. It runs the bounded GPU training and runtime pilot, adapts the screening episode target to measured P95 time, and publishes a development-only report. The launcher reserves 600 seconds for publication inside a 5,400-second execution budget; the notebook enforces a 7,200-second hard limit. Checkpoints and incomplete evidence remain in Kaggle Output. This does not run the frozen formal matrix or produce a scientific decision.

In [ ]:
from pathlib import Path
import os, signal, subprocess, sys, time

BRANCH = 'codex/resolve-epclm001-remote-branch-conflicts'
REPOSITORY = 'https://github.com/ArcheLabs/mini-cells.git'
WORKSPACE = Path('/kaggle/working')
REPO = WORKSPACE / 'minicells-epclm-002-a0'
HARD_LIMIT_SECONDS = 7200
deadline = time.monotonic() + HARD_LIMIT_SECONDS

def run_bounded(command):
    remaining = deadline - time.monotonic()
    if remaining <= 1:
        raise TimeoutError('The two-hour notebook budget is exhausted.')
    process = subprocess.Popen(command, cwd=WORKSPACE, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, start_new_session=(os.name == 'posix'))
    try:
        output, _ = process.communicate(timeout=remaining)
    except subprocess.TimeoutExpired:
        if os.name == 'posix':
            os.killpg(process.pid, signal.SIGTERM)
        else:
            process.terminate()
        try:
            output, _ = process.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            if os.name == 'posix':
                os.killpg(process.pid, signal.SIGKILL)
            else:
                process.kill()
            output, _ = process.communicate()
        print(output or '')
        raise TimeoutError('Hard two-hour limit reached. Kaggle Output remains under ' + str(REPO / 'results/epclm-002-a0'))
    print(output or '')
    if process.returncode != 0:
        raise RuntimeError('Command failed (' + str(process.returncode) + '). Kaggle Output remains under ' + str(REPO / 'results/epclm-002-a0'))

if REPO.exists():
    if not (REPO / '.git').exists():
        import shutil
        shutil.rmtree(REPO)
if REPO.exists():
    origin = subprocess.run(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], capture_output=True, text=True)
    if origin.returncode != 0 or 'ArcheLabs/mini-cells' not in origin.stdout:
        raise RuntimeError(f'Refusing to use an unrelated directory: {REPO}')
    run_bounded(['git', '-C', str(REPO), 'fetch', '--depth=1', 'origin', BRANCH])
    run_bounded(['git', '-C', str(REPO), 'reset', '--hard', 'FETCH_HEAD'])
else:
    run_bounded(['git', 'clone', '--depth=1', '--single-branch', '--branch', BRANCH, REPOSITORY, str(REPO)])

launcher = REPO / 'research/projects/cellular-language-models/experiments/epclm-002-a0-persistent-state-recall/scripts/kaggle_screening_launch.py'
run_bounded([sys.executable, str(launcher), '--max-runtime-seconds', '6000'])
print('Run finished. The v0.2 screening report was returned to GitHub; pilot timings, checkpoints, per-episode evidence, and incomplete cells remain in Kaggle Output.')